# Modelos de clasificación para aprendizaje automático relacional

En este notebook se entrenan y comparan varios modelos de clasificación para predecir la clase de los nodos del dataset Cora.

Se comparan tres conjuntos de características:

- **Features relacionales**: métricas calculadas a partir de la estructura del grafo.
- **Features nativas**: atributos propios de los nodos, representados como variables `word_*`.
- **Features combinadas**: unión de las features relacionales y las nativas.

Los modelos utilizados son:

- Árbol de decisión.
- KNN.
- Random Forest.

Además, se aplica ajuste de hiperparámetros mediante `GridSearchCV` sobre algunos modelos.

In [1]:
%pip install pandas numpy scikit-learn matplotlib
# ============================================
# 1. IMPORTS
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Carga del dataset

Se carga el fichero `features.csv`, que contiene una fila por cada nodo del grafo.  
Cada fila incluye métricas relacionales, atributos nativos del nodo y la clase que se quiere predecir.

In [3]:
# ============================================
# 2. CARGA DEL DATASET
# ============================================

df = pd.read_csv("../data/features.csv")

print("Tamaño del dataset:", df.shape)
df.head()

Tamaño del dataset: (2485, 1444)


,node_id,degree,closeness,betweenness,pagerank,eigenvector_centrality,clustering_coef,triangles,kcore,community_louvain,...,word_1424,word_1425,word_1426,word_1427,word_1428,word_1429,word_1430,word_1431,word_1432,class
0,1033,5,0.198023,0.002886,0.000450,0.052207,0.100000,1,3,0,...,0,0,0,0,0,0,0,0,0,Genetic_Algorithms
1,35,168,0.242768,0.276114,0.013302,0.654300,0.011406,160,4,1,...,0,0,0,0,0,0,0,0,0,Genetic_Algorithms
2,103482,6,0.202958,0.005357,0.000572,0.050281,0.133333,2,3,2,...,0,0,0,0,0,0,0,0,0,Neural_Networks
3,103515,11,0.196706,0.001996,0.000899,0.072508,0.163636,9,3,1,...,0,0,0,0,0,0,0,0,0,Genetic_Algorithms
4,1050679,4,0.225962,0.028597,0.000355,0.050567,0.166667,1,3,3,...,0,0,0,0,0,0,0,0,0,Genetic_Algorithms


## 2. Exploración básica

Antes de entrenar los modelos, se revisan las columnas disponibles, la distribución de clases y la existencia de valores nulos.

In [4]:
# ============================================
# 3. EXPLORACIÓN BÁSICA
# ============================================

print("Primeras columnas:")
print(df.columns[:15].tolist())

print("Número total de columnas:", len(df.columns))

print("Distribución de clases:")
print(df["class"].value_counts())

print("Valores nulos totales:", df.isnull().sum().sum())

Primeras columnas:
['node_id', 'degree', 'closeness', 'betweenness', 'pagerank', 'eigenvector_centrality', 'clustering_coef', 'triangles', 'kcore', 'community_louvain', 'word_0', 'word_1', 'word_2', 'word_3', 'word_4']
Número total de columnas: 1444
Distribución de clases:
class
Neural_Networks           726
Genetic_Algorithms        406
Probabilistic_Methods     379
Theory                    344
Case_Based                285
Reinforcement_Learning    214
Rule_Learning             131
Name: count, dtype: int64
Valores nulos totales: 0


## 3. Separación de características

Se separan las variables predictoras en tres grupos:

- **Relacionales**: métricas obtenidas a partir del grafo.
- **Nativas**: columnas `word_*`, que representan características propias de los artículos.
- **Combinadas**: unión de las anteriores.

La variable objetivo es `class`.

In [5]:
# ============================================
# 4. SEPARACIÓN DE FEATURES
# ============================================

target = "class"

features_relacionales = [
    "degree",
    "closeness",
    "betweenness",
    "pagerank",
    "eigenvector_centrality",
    "clustering_coef",
    "triangles",
    "kcore",
    "community_louvain"
]

features_nativas = [col for col in df.columns if col.startswith("word_")]

features_combinadas = features_relacionales + features_nativas

X_rel = df[features_relacionales]
X_nat = df[features_nativas]
X_comb = df[features_combinadas]

y = df[target]

print("Features relacionales:", X_rel.shape)
print("Features nativas:", X_nat.shape)
print("Features combinadas:", X_comb.shape)
print("Variable objetivo:", y.shape)

Features relacionales: (2485, 9)
Features nativas: (2485, 1433)
Features combinadas: (2485, 1442)
Variable objetivo: (2485,)


## 4. Función de entrenamiento y evaluación

Se define una función auxiliar para entrenar un modelo, evaluarlo sobre un conjunto de prueba y guardar sus resultados.

Las métricas utilizadas son:

- Accuracy.
- Precision macro.
- Recall macro.
- Matriz de confusión.

In [6]:
# ============================================
# 5. FUNCIÓN DE ENTRENAMIENTO Y EVALUACIÓN
# ============================================

def entrenar_y_evaluar(nombre_modelo, modelo, nombre_features, X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    matriz = confusion_matrix(y_test, y_pred)

    print("=" * 70)
    print("Modelo:", nombre_modelo)
    print("Features:", nombre_features)
    print("=" * 70)
    print("Accuracy:", accuracy)
    print("Precision macro:", precision)
    print("Recall macro:", recall)
    print("Matriz de confusión:")
    print(matriz)

    return {
        "modelo": nombre_modelo,
        "features": nombre_features,
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall
    }

## 5. Árbol de decisión

Primero se entrena un árbol de decisión con los tres conjuntos de características para comprobar qué tipo de información resulta más útil.

In [7]:
# ============================================
# 6. MODELO BASE: ÁRBOL DE DECISIÓN
# ============================================

resultados = []

resultados.append(entrenar_y_evaluar(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42),
    "relacionales",
    X_rel,
    y
))

resultados.append(entrenar_y_evaluar(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42),
    "nativas",
    X_nat,
    y
))

resultados.append(entrenar_y_evaluar(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42),
    "combinadas",
    X_comb,
    y
))

Modelo: Decision Tree
Features: relacionales
Accuracy: 0.7142857142857143
Precision macro: 0.6967640658712424
Recall macro: 0.6845512566644351
Matriz de confusión:
[[ 36   2   4   2   0   5   8]
 [  0  74   4   0   3   0   0]
 [  3   8 113   6   4   0  11]
 [  1   1  16  51   2   1   4]
 [  0   2   7   2  30   0   2]
 [  3   0   1   1   0  15   6]
 [  7   1  13   4   3   5  36]]
Modelo: Decision Tree
Features: nativas
Accuracy: 0.6338028169014085
Precision macro: 0.596362938291498
Recall macro: 0.5778765474317026
Matriz de confusión:
[[ 31   3   5   2   4   7   5]
 [  1  67   9   0   0   3   1]
 [  7  10 105   4   3   2  14]
 [  4   0  18  45   2   3   4]
 [  2   4   4   2  25   1   5]
 [  2   2   5   2   1   7   7]
 [  5   6  15   1   6   1  35]]
Modelo: Decision Tree
Features: combinadas
Accuracy: 0.7505030181086519
Precision macro: 0.7219671002420311
Recall macro: 0.701015536441887
Matriz de confusión:
[[ 36   0   8   1   2   4   6]
 [  0  75   1   0   4   0   1]
 [  1   5 121   7  

## 6. KNN

A continuación se entrena un clasificador KNN.  
Como KNN se basa en distancias, se utiliza `StandardScaler` dentro de un `Pipeline` para escalar las variables antes del entrenamiento.

In [8]:
# ============================================
# 7. MODELO BASE: KNN
# ============================================

resultados.append(entrenar_y_evaluar(
    "KNN",
    Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ]),
    "relacionales",
    X_rel,
    y
))

resultados.append(entrenar_y_evaluar(
    "KNN",
    Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ]),
    "nativas",
    X_nat,
    y
))

resultados.append(entrenar_y_evaluar(
    "KNN",
    Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ]),
    "combinadas",
    X_comb,
    y
))

Modelo: KNN
Features: relacionales
Accuracy: 0.5593561368209256
Precision macro: 0.5280906202098568
Recall macro: 0.5008897589395591
Matriz de confusión:
[[29  0 10 11  0  2  5]
 [ 0 72  9  0  0  0  0]
 [10  7 96 14  5  1 12]
 [ 7  1 23 30  6  0  9]
 [ 5  2  5  4 22  1  4]
 [ 4  1  8  1  3  5  4]
 [ 6  0 23  7  6  3 24]]
Modelo: KNN
Features: nativas
Accuracy: 0.4124748490945674
Precision macro: 0.5193530173815052
Recall macro: 0.35440863190797894
Matriz de confusión:
[[20  9 21  0  3  0  4]
 [ 2 67  9  0  1  0  2]
 [10 35 77  4  3  0 16]
 [14 14 22 11  1  0 14]
 [ 4  9 19  0 11  0  0]
 [ 1  5 12  1  1  4  2]
 [ 8 11 32  0  3  0 15]]
Modelo: KNN
Features: combinadas
Accuracy: 0.4386317907444668
Precision macro: 0.5439678500061765
Recall macro: 0.36449808796680033
Matriz de confusión:
[[20  7 24  0  2  0  4]
 [ 1 67  8  0  2  0  3]
 [ 9 29 91  4  5  0  7]
 [ 8  5 35 13  1  0 14]
 [ 4  8 20  0 10  0  1]
 [ 1  4 15  1  0  4  1]
 [ 7  3 44  0  2  0 13]]


## 7. Ajuste de hiperparámetros del árbol de decisión

Se aplica `GridSearchCV` sobre el árbol de decisión utilizando las features combinadas, ya que fueron las que dieron mejor resultado inicial.

In [9]:
# ============================================
# 8. AJUSTE DE HIPERPARÁMETROS: DECISION TREE
# ============================================

param_grid_arbol = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_arbol = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_arbol,
    cv=5,
    scoring="accuracy"
)

grid_arbol.fit(X_comb, y)

print("Mejores hiperparámetros:")
print(grid_arbol.best_params_)

print("Mejor accuracy medio en validación cruzada:")
print(grid_arbol.best_score_)

Mejores hiperparámetros:
{'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 10}
Mejor accuracy medio en validación cruzada:
0.7348088531187122


In [10]:
# ============================================
# 9. EVALUACIÓN DEL ÁRBOL OPTIMIZADO
# ============================================

mejor_arbol = DecisionTreeClassifier(
    max_depth=grid_arbol.best_params_["max_depth"],
    min_samples_leaf=grid_arbol.best_params_["min_samples_leaf"],
    min_samples_split=grid_arbol.best_params_["min_samples_split"],
    random_state=42
)

resultados.append(entrenar_y_evaluar(
    "Decision Tree GridSearch",
    mejor_arbol,
    "combinadas",
    X_comb,
    y
))

Modelo: Decision Tree GridSearch
Features: combinadas
Accuracy: 0.778672032193159
Precision macro: 0.7601758175179668
Recall macro: 0.736393422129318
Matriz de confusión:
[[ 36   0   4   1   2   1  13]
 [  1  77   2   0   1   0   0]
 [  1   4 121   8   2   0   9]
 [  0   0  11  60   1   1   3]
 [  0   3   3   1  33   2   1]
 [  3   0   0   1   0  13   9]
 [  8   0   6   2   4   2  47]]


## 8. Ajuste de hiperparámetros de KNN

Se aplica `GridSearchCV` sobre KNN usando las features relacionales, ya que fueron las que mejor funcionaron para este modelo en la evaluación inicial.

In [11]:
# ============================================
# 10. AJUSTE DE HIPERPARÁMETROS: KNN
# ============================================

pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    pipeline_knn,
    param_grid_knn,
    cv=5,
    scoring="accuracy"
)

grid_knn.fit(X_rel, y)

print("Mejores hiperparámetros:")
print(grid_knn.best_params_)

print("Mejor accuracy medio en validación cruzada:")
print(grid_knn.best_score_)

Mejores hiperparámetros:
{'knn__metric': 'manhattan', 'knn__n_neighbors': 11, 'knn__weights': 'distance'}
Mejor accuracy medio en validación cruzada:
0.552112676056338


In [12]:
# ============================================
# 11. EVALUACIÓN DEL KNN OPTIMIZADO
# ============================================

mejor_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=grid_knn.best_params_["knn__n_neighbors"],
        weights=grid_knn.best_params_["knn__weights"],
        metric=grid_knn.best_params_["knn__metric"]
    ))
])

resultados.append(entrenar_y_evaluar(
    "KNN GridSearch",
    mejor_knn,
    "relacionales",
    X_rel,
    y
))

Modelo: KNN GridSearch
Features: relacionales
Accuracy: 0.6056338028169014
Precision macro: 0.5862985216036296
Recall macro: 0.5787871729043127
Matriz de confusión:
[[26  0 10  9  1  3  8]
 [ 0 75  6  0  0  0  0]
 [10  8 94 11  6  1 15]
 [ 4  0 19 36  5  0 12]
 [ 3  2  4  3 28  1  2]
 [ 2  0  4  2  2 12  4]
 [ 4  0 17  7  6  5 30]]


## 9. Random Forest

Se entrena un Random Forest como modelo adicional.  
Este modelo combina varios árboles de decisión, lo que suele mejorar la capacidad de generalización frente a un único árbol.

In [13]:
# ============================================
# 12. MODELO BASE: RANDOM FOREST
# ============================================

resultados.append(entrenar_y_evaluar(
    "Random Forest",
    RandomForestClassifier(random_state=42),
    "relacionales",
    X_rel,
    y
))

resultados.append(entrenar_y_evaluar(
    "Random Forest",
    RandomForestClassifier(random_state=42),
    "nativas",
    X_nat,
    y
))

resultados.append(entrenar_y_evaluar(
    "Random Forest",
    RandomForestClassifier(random_state=42),
    "combinadas",
    X_comb,
    y
))

Modelo: Random Forest
Features: relacionales
Accuracy: 0.6841046277665996
Precision macro: 0.6559758047308587
Recall macro: 0.6440439485821935
Matriz de confusión:
[[ 35   0   5   3   0   4  10]
 [  0  79   1   0   1   0   0]
 [  6   8 109   8   4   0  10]
 [  1   0  18  46   4   0   7]
 [  1   4   4   3  28   2   1]
 [  3   0   5   1   0  12   5]
 [  8   0  17   4   2   7  31]]
Modelo: Random Forest
Features: nativas
Accuracy: 0.7545271629778671
Precision macro: 0.7831780123013223
Recall macro: 0.7006969361741486
Matriz de confusión:
[[ 41   1   9   0   2   0   4]
 [  1  70   8   1   0   0   1]
 [  1   4 132   1   0   0   7]
 [  6   0  17  49   1   0   3]
 [  2   6   6   0  26   0   3]
 [  2   0   3   1   1  14   5]
 [  4   2  14   2   1   3  43]]
Modelo: Random Forest
Features: combinadas
Accuracy: 0.7887323943661971
Precision macro: 0.8018045071343822
Recall macro: 0.7283431371524411
Matriz de confusión:
[[ 38   0   5   1   2   1  10]
 [  0  80   1   0   0   0   0]
 [  1   4 132   1

## 10. Ajuste de hiperparámetros de Random Forest

Como Random Forest con features combinadas obtuvo los mejores resultados iniciales, se aplica `GridSearchCV` para ajustar sus hiperparámetros.

In [ ]:
# ============================================
# 13. AJUSTE DE HIPERPARÁMETROS: RANDOM FOREST
# ============================================

param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring="accuracy"
)

grid_rf.fit(X_comb, y)

print("Mejores hiperparámetros:")
print(grid_rf.best_params_)

print("Mejor accuracy medio en validación cruzada:")
print(grid_rf.best_score_)

In [ ]:
# ============================================
# 14. EVALUACIÓN DEL RANDOM FOREST OPTIMIZADO
# ============================================

mejor_rf = RandomForestClassifier(
    max_depth=grid_rf.best_params_["max_depth"],
    min_samples_leaf=grid_rf.best_params_["min_samples_leaf"],
    min_samples_split=grid_rf.best_params_["min_samples_split"],
    n_estimators=grid_rf.best_params_["n_estimators"],
    random_state=42
)

resultados.append(entrenar_y_evaluar(
    "Random Forest GridSearch",
    mejor_rf,
    "combinadas",
    X_comb,
    y
))

## 11. Comparación final de resultados

Se construye una tabla final con todos los modelos evaluados, ordenada por accuracy.

In [ ]:
# ============================================
# 15. TABLA FINAL DE RESULTADOS
# ============================================

tabla_resultados = pd.DataFrame(resultados)

tabla_resultados = tabla_resultados.sort_values(
    by="accuracy",
    ascending=False
)

tabla_resultados

## 12. Conclusiones preliminares

El mejor modelo obtenido es Random Forest con ajuste de hiperparámetros mediante `GridSearchCV` y usando features combinadas.

Los resultados muestran que combinar las métricas relacionales del grafo con las características nativas de los nodos proporciona mejores resultados que usar cada grupo de características por separado.

KNN obtiene resultados claramente inferiores al árbol de decisión y a Random Forest, incluso tras el ajuste de hiperparámetros. Esto puede deberse a que KNN depende de distancias entre ejemplos y el problema tiene un número elevado de dimensiones, especialmente al usar las features nativas o combinadas.